# 🏠 House Price Prediction — Training Notebook

**End-to-end ML pipeline:**
1. Data loading & exploration (EDA)
2. Missing value imputation
3. Outlier detection & capping
4. Feature engineering
5. Encoding + scaling
6. Model training & comparison
7. Best-model selection & saving
8. Feature importance visualisation


In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# Make src/ importable from the notebook
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
print('Setup complete ✓')

## 1 · Load Data

In [ ]:
df = pd.read_csv('../data/housing.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Duplicates:', df.duplicated().sum())

In [ ]:
df.describe()

## 2 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sale price distribution
axes[0].hist(df['SalePrice'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Sale Price Distribution')
axes[0].set_xlabel('Sale Price ($)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))

# Correlation with SalePrice
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()['SalePrice'].drop('SalePrice').sort_values()
corr.plot(kind='barh', ax=axes[1], color=plt.cm.RdYlGn((corr - corr.min()) / (corr.max() - corr.min())))
axes[1].set_title('Feature Correlations with SalePrice')

# Living area vs price
sc = axes[2].scatter(df['GrLivArea'], df['SalePrice'], c=df['OverallQual'],
                     cmap='RdYlGn', alpha=0.5, s=15)
plt.colorbar(sc, ax=axes[2], label='Quality')
axes[2].set_title('Living Area vs Sale Price')
axes[2].set_xlabel('GrLivArea (sq ft)')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 3 · Preprocessing Pipeline

In [ ]:
from preprocess import HouseDataPreprocessor

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)}  |  Test: {len(test_df)}')

pre = HouseDataPreprocessor(k_features=20)
X_train, y_train = pre.fit_transform(train_df)
X_test,  y_test  = pre.transform(test_df)

print(f'\nFeature matrix — Train: {X_train.shape}, Test: {X_test.shape}')
print('Selected features:', pre.selected_features)

## 4 · Model Training & Comparison

In [ ]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

MODELS = {
    'LinearRegression':  LinearRegression(),
    'RandomForest':      RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'GradientBoosting':  GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
    'XGBoost':           XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbosity=0),
}

results = {}
for name, model in MODELS.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'MAE':  mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2':   r2_score(y_test, y_pred),
        'Time': round(time.time() - t0, 2),
    }

result_df = pd.DataFrame(results).T
result_df = result_df.sort_values('R2', ascending=False)
result_df.style.highlight_max(subset=['R2'], color='lightgreen') \
               .highlight_min(subset=['MAE','RMSE'], color='lightgreen') \
               .format({'MAE':'${:,.0f}','RMSE':'${:,.0f}','R2':'{:.4f}','Time':'{:.2f}s'})

In [ ]:
# Visualise model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_plot = ['MAE', 'RMSE', 'R2']
colors = ['#E84855', '#3BB273', '#2E86AB', '#F4A261']
names = list(results.keys())

for ax, metric in zip(axes, metrics_plot):
    vals = [results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors, edgecolor='white')
    ax.set_title(f'{metric} Comparison', fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{v:.3f}' if metric == 'R2' else f'${v:,.0f}',
                ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5 · Feature Importance

In [ ]:
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = MODELS[best_name]
print(f'Best model: {best_name}  R²={results[best_name]["R2"]:.4f}')

if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    feat_imp = pd.Series(imp, index=pre.selected_features).sort_values(ascending=True)[-15:]
    feat_imp.plot(kind='barh', figsize=(10, 6), color='steelblue')
    plt.title(f'Top-15 Feature Importances ({best_name})', fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Linear model — showing coefficient magnitudes')
    coefs = pd.Series(np.abs(best_model.coef_), index=pre.selected_features).sort_values(ascending=True)[-15:]
    coefs.plot(kind='barh', figsize=(10, 6), color='steelblue')
    plt.title('Top-15 |Coefficients| (LinearRegression)', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6 · Save Model

In [ ]:
import joblib
from preprocess import TARGET

artifact = {
    'model':             best_model,
    'preprocessor':      pre,
    'model_name':        best_name,
    'selected_features': pre.selected_features,
    'metrics':           results[best_name],
    'model_version':     '1.0.0',
    'all_results':       results,
}
joblib.dump(artifact, '../models/house_price_model.pkl')
print('Model saved ✓')

## 7 · Test Prediction

In [ ]:
from predict import predict_price

sample = {
    'overall_qual':  7,
    'gr_liv_area':   1800,
    'garage_cars':   2,
    'total_bsmt_sf': 1000,
    'full_bath':     2,
    'year_built':    2005,
}

price = predict_price(sample)
print(f'Predicted House Price: ${price:,.2f}')